In [1]:
# Figure and table generator for memoria_tfg.tex
# Run this notebook from top to bottom.

from pathlib import Path
import shutil
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path(r"c:\Universidad\TFG")
IMG_DIR = ROOT / "imagenes"
TAB_DIR = ROOT / "tablas_generadas"
IMG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

POS_METRICS_CSV = ROOT / "checkpoints" / "position_model" / "training_metrics.csv"
GAME_METRICS_CSV = ROOT / "checkpoints" / "game_models" / "training_metrics_games.csv"
POS_EVAL_DIR = ROOT / "checkpoints" / "position_model"

DATA_FINAL = ROOT / "data" / "final" / "annotated_positions_final.csv"
TRAIN_FINAL = ROOT / "data" / "final" / "train_df_final.csv"
VAL_FINAL = ROOT / "data" / "final" / "val_df_final.csv"
TEST_FINAL = ROOT / "data" / "final" / "test_df_final.csv"

plt.style.use("seaborn-v0_8-whitegrid")
print("ROOT:", ROOT)
print("IMG_DIR:", IMG_DIR)
print("TAB_DIR:", TAB_DIR)

ROOT: c:\Universidad\TFG
IMG_DIR: c:\Universidad\TFG\imagenes
TAB_DIR: c:\Universidad\TFG\tablas_generadas


In [2]:
# Utilities

def save_table(df: pd.DataFrame, stem: str):
    csv_path = TAB_DIR / f"{stem}.csv"
    tex_path = TAB_DIR / f"{stem}.tex"
    df.to_csv(csv_path, index=False, encoding="utf-8")
    with open(tex_path, "w", encoding="utf-8") as f:
        f.write(df.to_latex(index=False, escape=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))
    return csv_path, tex_path


def load_best_position_epoch(pos_metrics: pd.DataFrame) -> int:
    idx = pos_metrics["recall_at_10"].idxmax()
    return int(pos_metrics.loc[idx, "epoch"])


def load_best_game_epoch(game_metrics: pd.DataFrame) -> int:
    idx = game_metrics["recall_at_10"].idxmax()
    return int(game_metrics.loc[idx, "epoch"])


def pick_eval_file_for_epoch(epoch: int) -> Path:
    p = POS_EVAL_DIR / f"eval_metrics_by_distance_epoch_{epoch:04d}.csv"
    if p.exists():
        return p
    # Fallback if the exact file does not exist.
    cands = sorted(POS_EVAL_DIR.glob("eval_metrics_by_distance_epoch_*.csv"))
    if not cands:
        raise FileNotFoundError("No eval_metrics_by_distance_epoch_*.csv files were found")
    return cands[-1]


def parse_interval_key(s: str) -> float:
    s = str(s).strip()
    if s.startswith(">="):
        return float(s[2:])
    m = re.match(r"(\d+)\s*-\s*(\d+)", s)
    if m:
        a, b = map(int, m.groups())
        return (a + b) / 2.0
    return 1e9

print("Utilities loaded.")

Utilities loaded.


In [3]:
# Load base metrics
pos_metrics = pd.read_csv(POS_METRICS_CSV)
game_metrics = pd.read_csv(GAME_METRICS_CSV)

best_pos_epoch = load_best_position_epoch(pos_metrics)
best_game_epoch = load_best_game_epoch(game_metrics)

pos_eval_file = pick_eval_file_for_epoch(best_pos_epoch)
pos_eval = pd.read_csv(pos_eval_file)
pos_eval = pos_eval.sort_values("distance_interval", key=lambda s: s.map(parse_interval_key)).reset_index(drop=True)

print(f"Best position epoch: {best_pos_epoch}")
print(f"Best game epoch: {best_game_epoch}")
print(f"Distance eval file used: {pos_eval_file.name}")

Best position epoch: 6
Best game epoch: 7
Distance eval file used: eval_metrics_by_distance_epoch_0006.csv


In [5]:
# Figure 1: curvas_entrenamiento_posiciones.png
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
ax = axes.ravel()

ax[0].plot(pos_metrics["epoch"], pos_metrics["train_loss"], label="train_loss", lw=2)
ax[0].plot(pos_metrics["epoch"], pos_metrics["val_loss"], label="val_loss", lw=2)
ax[0].set_title("Positions: Loss by Epoch")
ax[0].set_xlabel("Epoch")
ax[0].legend()

ax[1].plot(pos_metrics["epoch"], pos_metrics["recall_at_1"], label="R@1", lw=2)
ax[1].plot(pos_metrics["epoch"], pos_metrics["recall_at_5"], label="R@5", lw=2)
ax[1].plot(pos_metrics["epoch"], pos_metrics["recall_at_10"], label="R@10", lw=2)
ax[1].set_title("Positions: Recall@k")
ax[1].set_xlabel("Epoch")
ax[1].legend()

ax[2].plot(pos_metrics["epoch"], pos_metrics["mrr"], label="MRR", lw=2, color="tab:green")
ax[2].set_title("Positions: MRR")
ax[2].set_xlabel("Epoch")
ax[2].legend()

ax[3].plot(pos_metrics["epoch"], pos_metrics["mean_rank"], label="Mean Rank", lw=2, color="tab:red")
ax[3].set_title("Positions: Mean Rank")
ax[3].set_xlabel("Epoch")
ax[3].legend()

fig.suptitle("Position Model Training", fontsize=14)
fig.tight_layout()
out_pos_curves = IMG_DIR / "curvas_entrenamiento_posiciones.png"
fig.savefig(out_pos_curves, dpi=200, bbox_inches="tight")
plt.close(fig)
print("Saved:", out_pos_curves)

# Figure 2: r10_vs_distancia.png
fig, ax = plt.subplots(figsize=(11, 5))
# Excluimos el primer intervalo (1-20) solo para este grafico
pos_eval_r10 = pos_eval[pos_eval["distance_interval"].astype(str).str.strip() != "1-20"].copy()
ax.bar(pos_eval_r10["distance_interval"], pos_eval_r10["recall_at_10"], color="tab:blue", alpha=0.9)
ax.set_title(f"Recall@10 by Distance Interval (Epoch {best_pos_epoch})")
ax.set_xlabel("Distance Interval")
ax.set_ylabel("Recall@10")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
out_r10 = IMG_DIR / "r10_vs_distancia.png"
fig.savefig(out_r10, dpi=200, bbox_inches="tight")
plt.close(fig)
print("Saved:", out_r10)

# Figure 3: curvas_entrenamiento_partidas.png
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
ax = axes.ravel()

ax[0].plot(game_metrics["epoch"], game_metrics["train_loss"], label="train_loss", lw=2)
ax[0].plot(game_metrics["epoch"], game_metrics["val_loss"], label="val_loss", lw=2)
ax[0].set_title("Games: Loss by Epoch")
ax[0].set_xlabel("Epoch")
ax[0].legend()

ax[1].plot(game_metrics["epoch"], game_metrics["recall_at_1"], label="R@1", lw=2)
ax[1].plot(game_metrics["epoch"], game_metrics["recall_at_5"], label="R@5", lw=2)
ax[1].plot(game_metrics["epoch"], game_metrics["recall_at_10"], label="R@10", lw=2)
ax[1].set_title("Games: Recall@k")
ax[1].set_xlabel("Epoch")
ax[1].legend()

ax[2].plot(game_metrics["epoch"], game_metrics["mrr"], label="MRR", lw=2, color="tab:green")
ax[2].set_title("Games: MRR")
ax[2].set_xlabel("Epoch")
ax[2].legend()

ax[3].plot(game_metrics["epoch"], game_metrics["mean_rank"], label="Mean Rank", lw=2, color="tab:red")
ax[3].set_title("Games: Mean Rank")
ax[3].set_xlabel("Epoch")
ax[3].legend()

fig.suptitle("Game Model Training", fontsize=14)
fig.tight_layout()
out_game_curves = IMG_DIR / "curvas_entrenamiento_partidas.png"
fig.savefig(out_game_curves, dpi=200, bbox_inches="tight")
plt.close(fig)
print("Saved:", out_game_curves)

# Figure 4: copy existing distance histogram
src_hist = ROOT / "distances_histogram.png"
dst_hist = IMG_DIR / "distances_histogram.png"
if src_hist.exists():
    shutil.copy2(src_hist, dst_hist)
    print("Copied:", dst_hist)
else:
    print("Not found:", src_hist)

Saved: c:\Universidad\TFG\imagenes\curvas_entrenamiento_posiciones.png
Saved: c:\Universidad\TFG\imagenes\r10_vs_distancia.png
Saved: c:\Universidad\TFG\imagenes\curvas_entrenamiento_partidas.png
Copied: c:\Universidad\TFG\imagenes\distances_histogram.png


In [4]:
# Tables for LaTeX
best_pos_row = pos_metrics.loc[pos_metrics["recall_at_10"].idxmax()].copy()
best_game_row = game_metrics.loc[game_metrics["recall_at_10"].idxmax()].copy()

# Table: metricas_globales_pos
metricas_globales_pos = pd.DataFrame({
    "Metric": ["Recall@1", "Recall@5", "Recall@10", "MRR", "Mean Rank"],
    "Value": [
        float(best_pos_row["recall_at_1"]),
        float(best_pos_row["recall_at_5"]),
        float(best_pos_row["recall_at_10"]),
        float(best_pos_row["mrr"]),
        float(best_pos_row["mean_rank"]),
    ],
})

# Table: metricas_intervalo_pos
metricas_intervalo_pos = pos_eval[[
    "distance_interval", "recall_at_1", "recall_at_5", "recall_at_10", "mrr", "mean_rank"
]].copy()
metricas_intervalo_pos = metricas_intervalo_pos.rename(columns={"distance_interval": "Interval"})

# Table: metricas_partidas
metricas_partidas = pd.DataFrame({
    "Metric": ["Recall@1", "Recall@5", "Recall@10", "MRR", "Mean Rank"],
    "Value": [
        float(best_game_row["recall_at_1"]),
        float(best_game_row["recall_at_5"]),
        float(best_game_row["recall_at_10"]),
        float(best_game_row["mrr"]),
        float(best_game_row["mean_rank"]),
    ],
})

# Table: comparacion_modelos
comparacion_modelos = pd.DataFrame({
    "Characteristic": [
        "Trainable parameters",
        "Visual backbone",
        "Text backbone",
        "Temperature",
        "Batch size",
        "Best R@10",
        "Best MRR",
    ],
    "Positions": [
        "~23M",
        "ResNet-18 (trainable)",
        "CLIP ViT-B/32 (partially)",
        "Learnable",
        "64",
        float(best_pos_row["recall_at_10"]),
        float(best_pos_row["mrr"]),
    ],
    "Games": [
        "~1.6M",
        "ResNet-18 (frozen)",
        "CLIP ViT-B/32 (frozen)",
        "Fixed (tau=0.07)",
        "256",
        float(best_game_row["recall_at_10"]),
        float(best_game_row["mrr"]),
    ],
})

# Table: stats_dataset_final
splits = {
    "annotated_positions_final": DATA_FINAL,
    "train_df_final": TRAIN_FINAL,
    "val_df_final": VAL_FINAL,
    "test_df_final": TEST_FINAL,
}
rows = []
for name, path in splits.items():
    d = pd.read_csv(path)
    rows.append({
        "Split": name,
        "Rows": int(len(d)),
        "Unique games": int(d["game_id"].nunique()) if "game_id" in d.columns else np.nan,
        "Unique FENs": int(d["fen"].nunique()) if "fen" in d.columns else np.nan,
    })
stats_dataset_final = pd.DataFrame(rows)

exported = {}
for stem, df in [
    ("metricas_globales_pos", metricas_globales_pos),
    ("metricas_intervalo_pos", metricas_intervalo_pos),
    ("metricas_partidas", metricas_partidas),
    ("comparacion_modelos", comparacion_modelos),
    ("stats_dataset_final", stats_dataset_final),
]:
    c, t = save_table(df, stem)
    exported[stem] = (c, t)

print("Exported tables:")
for k, (c, t) in exported.items():
    print(f"- {k}:\n   CSV={c}\n   TEX={t}")

display(metricas_globales_pos)
display(metricas_intervalo_pos.head())
display(metricas_partidas)
display(comparacion_modelos)
display(stats_dataset_final)

C:\Users\carlo\AppData\Local\Temp\ipykernel_45288\3751845142.py:75: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  d = pd.read_csv(path)


Exported tables:
- metricas_globales_pos:
   CSV=c:\Universidad\TFG\tablas_generadas\metricas_globales_pos.csv
   TEX=c:\Universidad\TFG\tablas_generadas\metricas_globales_pos.tex
- metricas_intervalo_pos:
   CSV=c:\Universidad\TFG\tablas_generadas\metricas_intervalo_pos.csv
   TEX=c:\Universidad\TFG\tablas_generadas\metricas_intervalo_pos.tex
- metricas_partidas:
   CSV=c:\Universidad\TFG\tablas_generadas\metricas_partidas.csv
   TEX=c:\Universidad\TFG\tablas_generadas\metricas_partidas.tex
- comparacion_modelos:
   CSV=c:\Universidad\TFG\tablas_generadas\comparacion_modelos.csv
   TEX=c:\Universidad\TFG\tablas_generadas\comparacion_modelos.tex
- stats_dataset_final:
   CSV=c:\Universidad\TFG\tablas_generadas\stats_dataset_final.csv
   TEX=c:\Universidad\TFG\tablas_generadas\stats_dataset_final.tex


,Metric,Value
0,Recall@1,0.133696
1,Recall@5,0.203215
2,Recall@10,0.248129
3,MRR,0.174387
4,Mean Rank,219.029794


,Interval,recall_at_1,recall_at_5,recall_at_10,mrr,mean_rank
0,1-20,0.076262,0.172932,0.234157,0.131269,198.944146
1,21-35,0.042000,0.079000,0.115000,0.068320,272.752000
2,36-50,0.061000,0.106000,0.144000,0.087688,260.837000
3,51-65,0.076000,0.139000,0.162000,0.110321,245.547000
4,66-80,0.100000,0.152000,0.194000,0.134689,231.355000


,Metric,Value
0,Recall@1,0.041298
1,Recall@5,0.094395
2,Recall@10,0.135693
3,MRR,0.078868
4,Mean Rank,107.330383


,Characteristic,Positions,Games
0,Trainable parameters,~23M,~1.6M
1,Visual backbone,ResNet-18 (trainable),ResNet-18 (frozen)
2,Text backbone,CLIP ViT-B/32 (partially),CLIP ViT-B/32 (frozen)
3,Temperature,Learnable,Fixed (tau=0.07)
4,Batch size,64,256
5,Best R@10,0.248129,0.135693
6,Best MRR,0.174387,0.078868


,Split,Rows,Unique games,Unique FENs
0,annotated_positions_final,219849,21203,203159
1,train_df_final,182915,11823,168615
2,val_df_final,10000,6042,9745
3,test_df_final,9000,5635,8786


In [ ]:
# Figure 5: diagrama_sistema_general.png (simple automatic generation)
fig, ax = plt.subplots(figsize=(14, 8))
ax.axis("off")

boxes = {
    "Data sources\n(PGN + GameKnot + Lichess)": (0.05, 0.70, 0.25, 0.16),
    "Cleaning and rewriting\npipeline": (0.37, 0.70, 0.22, 0.16),
    "Final dataset\n(position, text)": (0.68, 0.70, 0.22, 0.16),

    "Position model\n(ChessCLIP)": (0.18, 0.38, 0.26, 0.16),
    "Stratified evaluation\nby distance": (0.52, 0.38, 0.28, 0.16),

    "Game model\n(ChessGamesCLIP)": (0.18, 0.10, 0.26, 0.16),
    "Apps Streamlit\n(test_positions/test_games)": (0.52, 0.10, 0.34, 0.16),
}

for text, (x, y, w, h) in boxes.items():
    rect = plt.Rectangle((x, y), w, h, fill=True, alpha=0.15, edgecolor="black", linewidth=1.8)
    ax.add_patch(rect)
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=11)

# Arrows (correct flow)
arrow = dict(arrowstyle="->", lw=1.8)
ax.annotate("", xy=(0.37, 0.78), xytext=(0.30, 0.78), arrowprops=arrow)  # Sources -> Pipeline
ax.annotate("", xy=(0.68, 0.78), xytext=(0.59, 0.78), arrowprops=arrow)  # Pipeline -> Final dataset

ax.annotate("", xy=(0.36, 0.54), xytext=(0.74, 0.70), arrowprops=arrow)  # Final dataset -> Position model
ax.annotate("", xy=(0.66, 0.54), xytext=(0.79, 0.70), arrowprops=arrow)  # Final dataset -> Evaluation

ax.annotate("", xy=(0.31, 0.26), xytext=(0.31, 0.38), arrowprops=arrow)  # Position model -> Game model
ax.annotate("", xy=(0.52, 0.18), xytext=(0.44, 0.18), arrowprops=arrow)  # Game model -> Apps
ax.annotate("", xy=(0.69, 0.26), xytext=(0.66, 0.38), arrowprops=arrow)  # Evaluation -> Apps

ax.set_title("General ChessDualCLIP System Diagram", fontsize=14)
fig.tight_layout()
out_diagram = IMG_DIR / "diagrama_sistema_general.png"
fig.savefig(out_diagram, dpi=220, bbox_inches="tight")
plt.close(fig)
print("Saved:", out_diagram)

In [ ]:
# Final summary: what can and cannot be plotted automatically
plotables = [
    "diagrama_sistema_general.png",
    "distances_histogram.png (copied from project root)",
    "curvas_entrenamiento_posiciones.png",
    "r10_vs_distancia.png",
    "curvas_entrenamiento_partidas.png",
    "tables (.csv and .tex): metricas_globales_pos, metricas_intervalo_pos, metricas_partidas, comparacion_modelos, stats_dataset_final",
]

no_plotables_auto = [
    "ejemplo_busqueda_posicion.png (requires real screenshots from the running Streamlit app)",
    "ejemplo_busqueda_partida.png (requires real screenshots from the running Streamlit app)",
    "screenshots with slider, top-k, and concrete board queries (manual interaction required)",
]

print("=== AUTOMATICALLY GENERABLE ===")
for x in plotables:
    print("-", x)

print("\n=== NOT AUTOMATICALLY GENERABLE FROM STATIC DATA ===")
for x in no_plotables_auto:
    print("-", x)

print("\nImage output directory:", IMG_DIR)
print("Table output directory:", TAB_DIR)

In [ ]:
import chess
import chess.svg
from IPython.display import HTML, display

# Usamos exactamente el algoritmo de train_positions
from train_positions import board_to_planes_np, calculate_position_distance


def board_from_uci_sequence(uci_moves: list[str]) -> chess.Board:
    """Construye un tablero legal aplicando una secuencia de jugadas UCI."""
    board = chess.Board()
    for u in uci_moves:
        move = chess.Move.from_uci(u)
        if move not in board.legal_moves:
            raise ValueError(f"Jugada ilegal en la secuencia: {u}")
        board.push(move)
    return board


# Posicion A: estructura tipica de Ruy Lopez (normal y muy comun)
moves_a = [
    "e2e4", "e7e5",
    "g1f3", "b8c6",
    "f1b5", "a7a6",
    "b5a4", "g8f6",
    "e1g1", "f8e7",
    "f1e1", "b7b5",
    "a4b3", "d7d6",
    "c2c3", "e8g8",
    "h2h3",
]

# Posicion B: estructura tipica de Gambito de Dama rechazado
moves_b = [
    "d2d4", "d7d5",
    "c2c4", "e7e6",
    "b1c3", "g8f6",
    "c1g5", "f8e7",
    "e2e3", "e8g8",
    "g1f3", "h7h6",
    "g5h4", "b7b6",
    "c4d5", "e6d5",
]

board_a = board_from_uci_sequence(moves_a)
board_b = board_from_uci_sequence(moves_b)

planes_a = board_to_planes_np(board_a)
planes_b = board_to_planes_np(board_b)

distance = calculate_position_distance(planes_a, planes_b)

svg_a = chess.svg.board(board=board_a, size=360)
svg_b = chess.svg.board(board=board_b, size=360)

html = f"""
<div style='display:flex; gap:24px; align-items:flex-start; flex-wrap:wrap;'>
  <div>
    <h4 style='margin:0 0 8px 0;'>Posicion A (Ruy Lopez)</h4>
    {svg_a}
    <p style='font-family:monospace; font-size:12px; margin-top:8px;'>FEN: {board_a.fen()}</p>
  </div>
  <div>
    <h4 style='margin:0 0 8px 0;'>Posicion B (Gambito de Dama)</h4>
    {svg_b}
    <p style='font-family:monospace; font-size:12px; margin-top:8px;'>FEN: {board_b.fen()}</p>
  </div>
</div>
<p style='font-size:16px; margin-top:12px;'><b>Distancia (algoritmo train_positions): {distance}</b></p>
"""

display(HTML(html))